ABCDEDSkip scenario modification entirely. Only load Scenario A and B files.

Read scenario selections with pandas:

Read "SCENARIOS SELECTION" sheet.

Parse the “TABLE: …” and “X” marks directly in pandas.

Read GHG data with pandas only (as you already do).

Load data once and keep in memory.

Update GUI to remove the "Modify Scenarios" button.

In [5]:
import os
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import shutil
import tempfile

class FableDashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("FABLE Scenario Dashboard")
        self.scenario_paths = []
        self.temp_paths = []
        self.cols_to_plot = [
            "Calc CO2e from livestock",
            "Total calc. CO2e from crops",
            "Calc CO2 from deforestation",
            "Calc CO2 sequestration"
        ]
        self.scenario_colors = {"Scenario A": "blue", "Scenario B": "red"}
        self.sector_colors = {
            "Calc CO2e from livestock": "#636EFA",
            "Total calc. CO2e from crops": "#EF553B",
            "Calc CO2 from deforestation": "#00CC96",
            "Calc CO2 sequestration": "#AB63FA"
        }
        self.create_widgets()

    def create_widgets(self):
        tk.Label(self.root, text="Select Scenario A and B Files:").grid(row=0, column=0, padx=10, pady=5, sticky="w")
        tk.Button(self.root, text="Select Scenario Files", command=self.select_scenarios).grid(row=0, column=1, padx=5)

        self.visual_frame = tk.LabelFrame(self.root, text="Visualizations", padx=10, pady=10)
        self.visual_frame.grid(row=1, column=0, columnspan=2, padx=10, pady=10, sticky="ew")
        tk.Button(self.visual_frame, text="Yearly GHG by Sector", command=self.show_bar_chart).pack(pady=5)
        tk.Button(self.visual_frame, text="Total GHG Line Chart", command=self.show_total_ghg_chart).pack(pady=5)
        tk.Button(self.visual_frame, text="Combined Charts", command=self.show_combined_chart).pack(pady=5)

        self.summary_text = tk.Text(self.root, width=100, height=10, state="disabled")
        self.summary_text.grid(row=2, column=0, columnspan=2, padx=10, pady=10)

    # --- File selection ---
    def select_scenarios(self):
        paths = filedialog.askopenfilenames(title="Select Scenario A then B", filetypes=[("Excel files", "*.xlsx *.xlsm *.xls")])
        if len(paths) != 2:
            messagebox.showerror("Error", "Please select exactly two scenario files.")
            return

        # Copy to temp folder to avoid permission issues
        self.temp_paths = []
        for path in paths:
            tmp_path = shutil.copy(path, tempfile.gettempdir())
            self.temp_paths.append(tmp_path)

        self.scenario_paths = list(paths)
        self.update_summary()
        messagebox.showinfo("Loaded", "Scenario files loaded successfully!")

    # --- Scenario summary ---
    def update_summary(self):
        self.summary_text.config(state="normal")
        self.summary_text.delete("1.0", tk.END)
        for idx, path in enumerate(self.scenario_paths):
            label = f"Scenario {'A' if idx==0 else 'B'}"
            temp_path = self.temp_paths[idx]
            try:
                df = pd.read_excel(temp_path, sheet_name="SCENARIOS SELECTION", header=None)
                summary = []
                current_table = None
                for _, row in df.iterrows():
                    first = str(row[0]).strip() if row[0] else ""
                    second = str(row[1]).strip() if len(row) > 1 and row[1] else ""
                    if first.startswith("TABLE:"):
                        current_table = first.replace("TABLE:", "").strip()
                        continue
                    if first.lower() == "x" and current_table:
                        summary.append(f"{current_table}: {second}")
                self.summary_text.insert(tk.END, f"{label} ({os.path.basename(path)}):\n")
                for line in summary:
                    self.summary_text.insert(tk.END, f"  {line}\n")
                self.summary_text.insert(tk.END, "\n")
            except:
                self.summary_text.insert(tk.END, f"{label}: Could not read scenario selections\n")
        self.summary_text.config(state="disabled")

    # --- Load GHG data ---
    def load_ghg(self, idx):
        path = self.temp_paths[idx]
        df_raw = pd.read_excel(path, sheet_name="GHG", header=None)
        year_row = next((i for i in range(len(df_raw)) if df_raw.iloc[i].astype(str).str.strip().str.lower().eq("year").any()), None)
        if year_row is None:
            raise ValueError(f"No 'Year' header in {path}")
        df = pd.read_excel(path, sheet_name="GHG", skiprows=year_row)
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        for c in df.columns:
            if c != "Year":
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df[df["Year"].between(2000, 2050)]
        df = df.dropna(subset=["Year"])
        df["Year"] = df["Year"].astype(int)
        return df

    # --- Prepare combined data ---
    def prepare_combined_data(self):
        df_A = self.load_ghg(0)
        df_B = self.load_ghg(1)
        def make_long(df, label):
            df_long = df.melt(
                id_vars="Year",
                value_vars=[c for c in self.cols_to_plot if c in df.columns],
                var_name="Category",
                value_name="MtCO2e/yr"
            )
            df_long["Scenario"] = label
            return df_long
        df_combined = pd.concat([make_long(df_A, "Scenario A"), make_long(df_B, "Scenario B")])
        df_combined["Total GHG"] = df_combined.groupby(["Scenario","Year"])["MtCO2e/yr"].transform("sum")
        return df_combined

    # --- Helper to save HTML ---
    def save_html(self, fig, filename):
        folder = os.path.dirname(self.scenario_paths[0])
        path = os.path.join(folder, filename)
        fig.write_html(path)
        messagebox.showinfo("Saved", f"Interactive HTML saved to:\n{path}")

    # --- Plotting ---
    def show_bar_chart(self):
        if not self.scenario_paths:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        fig = px.bar(
            df_combined,
            x="Year",
            y="MtCO2e/yr",
            color="Category",
            barmode="group",
            facet_col="Scenario",
            color_discrete_map=self.sector_colors,
            title="Yearly GHG Emissions by Sector"
        )
        fig.show()
        self.save_html(fig, "Yearly_GHG_by_Sector.html")

    def show_total_ghg_chart(self):
        if not self.scenario_paths:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        fig = px.line(
            df_combined.drop_duplicates(subset=["Year","Scenario"]),
            x="Year",
            y="Total GHG",
            color="Scenario",
            color_discrete_map=self.scenario_colors,
            title="Total GHG Emissions (All Sectors)"
        )
        fig.show()
        self.save_html(fig, "Total_GHG_Line.html")

    def show_combined_chart(self):
        if not self.scenario_paths:
            messagebox.showwarning("Warning", "No scenario files loaded.")
            return
        df_combined = self.prepare_combined_data()
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=("Yearly GHG by Sector", "Total GHG Emissions"))
        for scenario in ["Scenario A", "Scenario B"]:
            df_s = df_combined[df_combined["Scenario"]==scenario]
            for category in self.cols_to_plot:
                df_cat = df_s[df_s["Category"]==category]
                fig.add_trace(
                    go.Bar(x=df_cat["Year"], y=df_cat["MtCO2e/yr"],
                           name=f"{category} ({scenario})",
                           marker_color=self.sector_colors.get(category, "#888888")),
                    row=1, col=1
                )
            df_total = df_s.drop_duplicates(subset=["Year"])
            fig.add_trace(
                go.Scatter(x=df_total["Year"], y=df_total["Total GHG"],
                           name=f"Total GHG ({scenario})",
                           line=dict(color=self.scenario_colors[scenario], width=3)),
                row=2, col=1
            )
        fig.update_layout(height=800, title_text="FABLE Scenario GHG Dashboard", barmode="group")
        fig.show()
        self.save_html(fig, "Combined_GHG_Charts.html")

# --- Run dashboard ---
root = tk.Tk()
app = FableDashboard(root)
root.mainloop()


KeyboardInterrupt: 